## Importing Packages

In [19]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Regression Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub
import os

## DagsHub MLflow Setup

In [20]:
import dagshub

dagshub.init(
    repo_owner="RidhiRajesh",
    repo_name="MLflow-101",
    mlflow=True
)

Initialized MLflow to track repo "RidhiRajesh/MLflow-101"

Repository RidhiRajesh/MLflow-101 initialized!

In [21]:
mlflow.set_experiment(
    "Housing Price Prediction PBLM 1"
)

<Experiment: artifact_location='mlflow-artifacts:/14ca32342eec4a5a876b08f7795910f7', creation_time=1786003065951, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786003065951, lifecycle_stage='active', name='Housing Price Prediction PBLM 1', tags={}, trace_location=None, workspace='default'>

## Data Loading and Processing

In [22]:
import pandas as pd

data = pd.read_csv(r"C:\Users\Ridhi Rajesh\Downloads\HousingData.csv")

# Remove missing values
data = data.dropna()

# Features and Target
X = data.drop("MEDV", axis=1)
y = data["MEDV"]

print("Shape:", X.shape)
print("Target Column: MEDV")
print("Target Statistics:")
print(y.describe())

Shape: (394, 13)
Target Column: MEDV
Target Statistics:
count    394.000000
mean      22.359645
std        9.142979
min        5.000000
25%       16.800000
50%       21.050000
75%       25.000000
max       50.000000
Name: MEDV, dtype: float64


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

## Build Models

In [24]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train,
        y_train
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost",
        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            random_state=42
        ),
        X_train,
        y_train
    )
]

In [25]:
reports = []
trained_models = []

for model_name, model, X_tr, y_tr in models:

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    report = {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

    reports.append(report)
    trained_models.append(model)

    print("=" * 50)
    print(model_name)
    print("=" * 50)
    print(f"MAE : {mae:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")

Linear Regression
MAE : 3.4558
MSE : 28.8708
RMSE: 5.3732
R²  : 0.6905
Random Forest
MAE : 2.5791
MSE : 20.9917
RMSE: 4.5817
R²  : 0.7750
XGBoost
MAE : 2.5432
MSE : 20.1651
RMSE: 4.4906
R²  : 0.7838


## Log All Experiments to DagsHub

In [26]:
for i, (model_name, model, _, _) in enumerate(models):

    report = reports[i]

    with mlflow.start_run(run_name=model_name):

        # Parameters
        mlflow.log_param("Model", model_name)
        mlflow.log_params(model.get_params())

        # Regression Metrics
        mlflow.log_metric("MAE", report["MAE"])
        mlflow.log_metric("MSE", report["MSE"])
        mlflow.log_metric("RMSE", report["RMSE"])
        mlflow.log_metric("R2", report["R2"])

        # Log Model
        if "XGBoost" in model_name:

            mlflow.xgboost.log_model(
                model,
                "model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                "model"
            )

print("Experiments logged successfully!")

2026/08/06 19:06:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/RidhiRajesh/MLflow-101.mlflow/#/experiments/0/runs/1699b427618f4db6a3900260038ca37d
🧪 View experiment at: https://dagshub.com/RidhiRajesh/MLflow-101.mlflow/#/experiments/0


2026/08/06 19:07:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: https://dagshub.com/RidhiRajesh/MLflow-101.mlflow/#/experiments/0/runs/640872e3fcc34a76b3d935cf504ed01f
🧪 View experiment at: https://dagshub.com/RidhiRajesh/MLflow-101.mlflow/#/experiments/0


2026/08/06 19:07:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: https://dagshub.com/RidhiRajesh/MLflow-101.mlflow/#/experiments/0/runs/fadcaf71cd4646aab2d94c9422b48ca9
🧪 View experiment at: https://dagshub.com/RidhiRajesh/MLflow-101.mlflow/#/experiments/0
Experiments logged successfully!


## Best Model and Reg to DH

In [28]:
best_index = np.argmax(
    [r["R2"] for r in reports]
)

best_model_name = models[best_index][0]
best_model = trained_models[best_index]
best_report = reports[best_index]

print("Best Model:", best_model_name)
print("R2 Score :", best_report["R2"])
print("MAE      :", best_report["MAE"])
print("MSE      :", best_report["MSE"])
print("RMSE     :", best_report["RMSE"])

Best Model: XGBoost
R2 Score : 0.7838392024277021
MAE      : 2.5432272149735136
MSE      : 20.16505174788402
RMSE     : 4.490551385730267


In [30]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:

    # Parameters
    mlflow.log_param(
        "Model",
        best_model_name
    )

    mlflow.log_param(
        "Selection_Metric",
        "R2 Score"
    )

    # Metrics
    mlflow.log_metric(
        "R2",
        best_report["R2"]
    )

    mlflow.log_metric(
        "MAE",
        best_report["MAE"]
    )

    mlflow.log_metric(
        "MSE",
        best_report["MSE"]
    )

    mlflow.log_metric(
        "RMSE",
        best_report["RMSE"]
    )

    # Log model
    if "XGBoost" in best_model_name:

        model_info = mlflow.xgboost.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Housing_Price_Best_Model"
        )

    else:

        model_info = mlflow.sklearn.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Housing_Price_Best_Model"
        )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print("Run ID    :", run_id)
print("Model URI :", model_uri)
print("Model Name:", "Housing_Price_Best_Model")

2026/08/06 19:11:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Housing_Price_Best_Model'.
2026/08/06 19:12:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Housing_Price_Best_Model, version 1
Created version '1' of model 'Housing_Price_Best_Model'.


🏃 View run Champion_XGBoost at: https://dagshub.com/RidhiRajesh/MLflow-101.mlflow/#/experiments/0/runs/fe376a775d3b4095aa78814d6f199840
🧪 View experiment at: https://dagshub.com/RidhiRajesh/MLflow-101.mlflow/#/experiments/0
Run ID    : fe376a775d3b4095aa78814d6f199840
Model URI : models:/m-454cd25ad3dc4ee1b7a2a65a45affdca
Model Name: Housing_Price_Best_Model
